# 11 · 하이브리드 실시간 — 정확도 + 부드러운 화면 (GPU 상시 활용)

문제: DA+FastSAM은 정확하지만 무거워서 동기 실행 시 ~1fps(끊김). 순수 경량(10)은 빠르지만 인식이 약함.

**해법(하이브리드):** 무거운 정확 파이프라인을 **백그라운드 스레드(GPU)** 에서 계속 돌리고, 메인 루프는 **웹캠을 매 프레임 부드럽게** 표시하며 최신 검출을 덧그린다.
- 화면 fps = 카메라 속도(부드러움)
- 검출 갱신 = 추론 속도(초당 1~수 회) — **GPU를 백그라운드에서 상시 사용**
- 물체를 놓고 확인하는 용도에 적합(검출 오버레이가 1회 추론만큼 지연될 수 있음).

> 속도 개선: DA 후처리(픽셀당 3D계산)를 numpy→**GPU(torch)** 로 옮겨 height_map 547ms→~260ms로 단축(정확도 동일).

| 모드 | 인식 | 화면 |
|---|---|---|
| 08/09 동기 | 정확 | ~1fps(끊김) |
| 10 경량 | 약함 | ~8~14fps |
| **11 하이브리드** | **정확** | **부드러움(카메라 fps)** |

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
import cv2, numpy as np
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, depth_volume as dv, live_hybrid as lh
from ultralytics import FastSAM

OUTPUT_DIR = os.path.join(ROOT, 'output')
SQ = 0.038
board = cv2.aruco.CharucoBoard((5, 7), SQ, SQ*22/30, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_1000))
K, dist = au.load_intrinsics(os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz'))
CAM_INDEX = 0
pipe = dv.load_depth_model('depth-anything/Depth-Anything-V2-Small-hf', device=0)
model = FastSAM('FastSAM-s.pt')
print('ready')

## 실시간 (하이브리드)

보드 위에 물체를 올리면, 화면은 부드럽게 흐르고 검출(윤곽·측정)이 초당 몇 번씩 갱신된다.
- **`s`**: 스냅  ·  **`q`**: 종료
- 상단에 `infer NNms (X Hz)` = 백그라운드 추론 속도 표시.
- 검출 갱신이 더 빠르길 원하면 `imgsz`↓ 또는 카메라를 고정.

In [ ]:
lh.run_live_hybrid(K, dist, board, pipe=pipe, model=model, square_len=SQ,
                   cam_index=CAM_INDEX, imgsz=640, snapshot_dir=OUTPUT_DIR)
print('종료됨')

## 정리

- **화면 부드러움**은 카메라 fps, **인식 정확도**는 DA+FastSAM 수준을 유지 — 둘 다 확보.
- 검출 오버레이는 최신 프레임보다 1회 추론만큼 지연될 수 있음(카메라 크게 움직이면 티남). 물체 배치·확인엔 무방.
- 더 빠른 갱신: `imgsz`↓(예: 480), 해상도↓. 더 정확: `imgsz`↑.
- 향후: 검출 결과를 물체 ID로 추적해 오버레이 지연 보정, 카메라 포즈로 오버레이 재투영.